# NavLoRI — Async Multi-Modal Data Collection Report
**Project:** Indoor Position Prediction using 4-Modality Fusion (WiFi + Vision + IMU + Odometry)  
**Robot:** TIAGO++ (Webots Simulation) | **Lab:** CESI LINEACT  
**Author:** Mohamed Bachar  

---

## Dataset Overview
- **5 modalities** collected asynchronously at independent rates with ±20% jitter
- **30 planned paths** through a realistic indoor environment (75 waypoints)
- **Collision-free** trajectories with safety margins (robot radius × 1.5)

| Modality | Nominal Rate | Description |
|----------|-------------|-------------|
| IMU | ~31 Hz | Accelerometer + Gyroscope + Orientation |
| Odometry | ~15 Hz | Wheel encoders → dead reckoning |
| Ground Truth | ~10 Hz | Supervisor API (x, y, z, heading) |
| WiFi RSSI | ~1 Hz | 117 access points, GPR-based predictor |
| Camera | ~0.5 Hz | RGB + Depth (640×480 PNG) |

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

# --- Load all collected paths ---
DATA_DIR = Path(r"C:\Users\Administrateur\navlori\fusion\data\async_collection")
PATHS_JSON = Path(r"C:\Users\Administrateur\navlori\simulation\controllers\async_collector\paths.json")
TOTAL_PLANNED_PATHS = 30

path_dirs = sorted(DATA_DIR.glob("path_*"))
# Exclude incomplete paths (empty CSVs or missing camera data)
complete_paths = []
for p in path_dirs:
    cam_csv = p / "camera.csv"
    if not cam_csv.exists() or cam_csv.stat().st_size < 50:
        continue
    try:
        if pd.read_csv(cam_csv).shape[0] > 1:
            complete_paths.append(p)
    except Exception:
        continue

N_COLLECTED = len(complete_paths)
print(f"Collected paths: {N_COLLECTED} / {TOTAL_PLANNED_PATHS}")
print(f"Path directories: {[p.name for p in complete_paths]}")

In [ ]:
# --- Load all CSVs into per-modality DataFrames ---
all_imu, all_odom, all_wifi, all_gt, all_cam = [], [], [], [], []
path_stats = []

for p in complete_paths:
    pid = int(p.name.split("_")[1])
    
    imu = pd.read_csv(p / "imu.csv");       imu["path_id"] = pid
    odom = pd.read_csv(p / "odometry.csv");  odom["path_id"] = pid
    wifi = pd.read_csv(p / "wifi.csv");      wifi["path_id"] = pid
    gt = pd.read_csv(p / "ground_truth.csv"); gt["path_id"] = pid
    cam = pd.read_csv(p / "camera.csv");     cam["path_id"] = pid
    
    all_imu.append(imu)
    all_odom.append(odom)
    all_wifi.append(wifi)
    all_gt.append(gt)
    all_cam.append(cam)
    
    duration = gt["sim_time"].max() - gt["sim_time"].min()
    n_images = len(list((p / "camera").glob("rgb_*.png")))
    disk_mb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
    
    path_stats.append({
        "path_id": pid, "duration_s": round(duration, 1),
        "imu_events": len(imu), "odom_events": len(odom),
        "wifi_scans": len(wifi), "gt_events": len(gt),
        "camera_frames": len(cam), "rgb_images": n_images,
        "disk_mb": round(disk_mb, 1),
    })

df_imu = pd.concat(all_imu, ignore_index=True)
df_odom = pd.concat(all_odom, ignore_index=True)
df_wifi = pd.concat(all_wifi, ignore_index=True)
df_gt = pd.concat(all_gt, ignore_index=True)
df_cam = pd.concat(all_cam, ignore_index=True)
df_stats = pd.DataFrame(path_stats)

print(f"Loaded {N_COLLECTED} paths:")
print(f"  IMU:     {len(df_imu):>8,} events")
print(f"  Odom:    {len(df_odom):>8,} events")
print(f"  WiFi:    {len(df_wifi):>8,} events")
print(f"  GT:      {len(df_gt):>8,} events")
print(f"  Camera:  {len(df_cam):>8,} frames")
print(f"  Disk:    {df_stats['disk_mb'].sum():>8.0f} MB")

## 1. Collection Summary & Projections (30 paths)

In [ ]:
# --- Summary table with projection to 30 paths ---
scale = TOTAL_PLANNED_PATHS / N_COLLECTED

summary = {
    "Modality": ["IMU", "Odometry", "WiFi RSSI", "Ground Truth", "Camera (RGB+Depth)"],
    "Rate": ["~31 Hz", "~15 Hz", "~1 Hz", "~10 Hz", "~0.5 Hz"],
    f"Collected ({N_COLLECTED} paths)": [
        f"{len(df_imu):,}", f"{len(df_odom):,}", f"{len(df_wifi):,}",
        f"{len(df_gt):,}", f"{len(df_cam):,}"
    ],
    f"Projected (30 paths)": [
        f"{int(len(df_imu)*scale):,}", f"{int(len(df_odom)*scale):,}",
        f"{int(len(df_wifi)*scale):,}", f"{int(len(df_gt)*scale):,}",
        f"{int(len(df_cam)*scale):,}"
    ],
}

df_summary = pd.DataFrame(summary)

# Add totals row
total_collected = len(df_imu) + len(df_odom) + len(df_wifi) + len(df_gt) + len(df_cam)
df_summary.loc[len(df_summary)] = [
    "**TOTAL**", "—",
    f"{total_collected:,}", f"{int(total_collected * scale):,}"
]

# Disk projection
disk_collected = df_stats["disk_mb"].sum()
df_summary.loc[len(df_summary)] = [
    "**Disk Size**", "—",
    f"{disk_collected:,.0f} MB", f"~{disk_collected * scale / 1000:.1f} GB"
]

avg_duration = df_stats["duration_s"].mean()
df_summary.loc[len(df_summary)] = [
    "**Avg Path Duration**", "—",
    f"{avg_duration:.0f} s", f"{avg_duration:.0f} s"
]

df_summary.style.set_properties(**{"text-align": "right"}).set_properties(
    subset=["Modality"], **{"text-align": "left", "font-weight": "bold"}
)

In [ ]:
# --- Events per path (grouped bar chart) ---
fig, ax = plt.subplots(figsize=(14, 5))

bar_data = df_stats.melt(
    id_vars="path_id",
    value_vars=["imu_events", "odom_events", "gt_events", "wifi_scans", "camera_frames"],
    var_name="modality", value_name="events"
)
bar_data["modality"] = bar_data["modality"].map({
    "imu_events": "IMU", "odom_events": "Odometry", "gt_events": "Ground Truth",
    "wifi_scans": "WiFi", "camera_frames": "Camera"
})

sns.barplot(data=bar_data, x="path_id", y="events", hue="modality", ax=ax)
ax.set_title("Sensor Events per Path", fontsize=14, fontweight="bold")
ax.set_xlabel("Path ID")
ax.set_ylabel("Number of Events")
ax.legend(title="Modality", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 2. Robot Trajectories (Ground Truth)

In [ ]:
# --- Wall geometry helper ---
import math as _math

RAW_WALLS = [
    (1.48, -1.47, 2.2327, 0.2, 2.7),
    (1.27, -3.35, -2.5454, 0.2, 2.707),
    (-0.61, -6.52, 2.2301, 0.2, 6.9709),
    (-10.33, -5.40, 2.15, 0.2, 7.5),
    (-2.16, -10.77, -2.6475, 0.2, 5.0),
    (1.10, -11.53, -1.0, 0.2, 5.1671),
    (3.99, -11.34, -2.6168, 0.2, 3.0),
    (-1.06, -16.10, 2.1194, 0.2, 13.8),
    (-8.01, -18.25, 0.6777, 0.2, 1.9),
    (-6.55, -16.13, -1.0151, 0.2, 5.1559),
    (-5.62, -12.74, 0.5114, 0.2, 4.6),
    (-8.43, -11.76, 2.1214, 0.2, 3.8606),
    (-12.77, -8.61, 0.5831, 0.2, 2.9),
    (-10.79, -11.60, 0.5831, 0.2, 2.7),
    (-6.30, -4.47, -2.482, 0.2, 3.1004),
    (-4.05, -4.58, -0.8897, 0.2, 3.6),
    (-3.72, -1.98, 0.5912, 0.2, 3.4447),
    (-2.75, 1.41, -0.8125, 0.2, 6.0),
    (0.84, 1.52, -2.5178, 0.2, 5.3),
]

def wall_polygons(inflate=0.0):
    polys = []
    for tx, ty, angle, sx, sy in RAW_WALLS:
        hx = sx / 2 + inflate
        hy = sy / 2 + inflate
        corners_local = [(-hx, -hy), (hx, -hy), (hx, hy), (-hx, hy)]
        ca, sa = _math.cos(angle), _math.sin(angle)
        corners = [(tx + ca*cx - sa*cy, ty + sa*cx + ca*cy) for cx, cy in corners_local]
        polys.append(np.array(corners))
    return polys

def draw_walls(ax, inflate=0.0, color="#b0b0b0", alpha=0.6):
    """Matplotlib helper."""
    for poly in wall_polygons(inflate):
        p = plt.Polygon(poly, closed=True, facecolor=color, edgecolor="#555", linewidth=0.5, alpha=alpha)
        ax.add_patch(p)

def plotly_wall_traces(showlegend_once=True):
    """Return scatter traces for walls (works in subplots)."""
    traces = []
    for i, poly in enumerate(wall_polygons()):
        xs = list(poly[:, 0]) + [poly[0, 0]]
        ys = list(poly[:, 1]) + [poly[0, 1]]
        traces.append(go.Scatter(
            x=xs, y=ys, fill="toself",
            fillcolor="rgba(180,180,180,0.45)",
            line=dict(color="rgba(100,100,100,0.6)", width=1),
            showlegend=(i == 0 and showlegend_once),
            name="Walls" if i == 0 else "",
            hoverinfo="skip",
        ))
    return traces

# --- Grid of individual paths (Plotly) ---
n_paths = len(df_stats)
ncols = min(4, n_paths)
nrows = _math.ceil(n_paths / ncols)

fig = make_subplots(rows=nrows, cols=ncols,
                    subplot_titles=[f"Path {pid}" for pid in df_stats["path_id"]],
                    horizontal_spacing=0.04, vertical_spacing=0.06)

colors = px.colors.qualitative.Vivid + px.colors.qualitative.Bold

for i, pid in enumerate(df_stats["path_id"]):
    r = i // ncols + 1
    c = i % ncols + 1
    path_gt = df_gt[df_gt["path_id"] == pid]

    # Walls
    for wt in plotly_wall_traces(showlegend_once=False):
        fig.add_trace(wt.update(fillcolor="rgba(200,200,200,0.4)"), row=r, col=c)

    # Trajectory
    fig.add_trace(go.Scatter(
        x=path_gt["gt_x"], y=path_gt["gt_y"],
        mode="lines", line=dict(width=2.5, color=colors[i % len(colors)]),
        showlegend=False, hoverinfo="skip",
    ), row=r, col=c)

    # Start / End
    fig.add_trace(go.Scatter(
        x=[path_gt["gt_x"].iloc[0]], y=[path_gt["gt_y"].iloc[0]],
        mode="markers", marker=dict(size=8, color="#4CAF50", symbol="circle"),
        showlegend=False, hovertemplate="Start<extra></extra>",
    ), row=r, col=c)
    fig.add_trace(go.Scatter(
        x=[path_gt["gt_x"].iloc[-1]], y=[path_gt["gt_y"].iloc[-1]],
        mode="markers", marker=dict(size=8, color="#F44336", symbol="x"),
        showlegend=False, hovertemplate="End<extra></extra>",
    ), row=r, col=c)

# Subplot borders + no inner grid
fig.update_xaxes(range=[-14, 5], showgrid=False, zeroline=False, showticklabels=False,
                 showline=True, linewidth=1.5, linecolor="#888", mirror=True)
fig.update_yaxes(range=[-20, 3], scaleanchor="x", showgrid=False, zeroline=False, showticklabels=False,
                 showline=True, linewidth=1.5, linecolor="#888", mirror=True)
fig.update_layout(
    title_text="Individual Robot Trajectories with Environment Layout",
    title_font_size=16,
    height=350 * nrows, width=280 * ncols,
    template="plotly_white",
    showlegend=False,
    plot_bgcolor="white",
)
fig.show()

In [ ]:
# --- Spatial coverage: KDE heatmap with env walls (matplotlib) ---
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(10, 8))
draw_walls(ax, color="#cccccc", alpha=0.8)

xy = np.vstack([df_gt["gt_x"].values, df_gt["gt_y"].values])
kde = gaussian_kde(xy, bw_method=0.15)

xmin, xmax = df_gt["gt_x"].min() - 1, df_gt["gt_x"].max() + 1
ymin, ymax = df_gt["gt_y"].min() - 1, df_gt["gt_y"].max() + 1
xg, yg = np.mgrid[xmin:xmax:200j, ymin:ymax:200j]
positions = np.vstack([xg.ravel(), yg.ravel()])
density = kde(positions).reshape(xg.shape)

im = ax.imshow(density.T, origin="lower", aspect="equal",
               extent=[xmin, xmax, ymin, ymax],
               cmap="magma_r", alpha=0.85)
plt.colorbar(im, ax=ax, label="Sample Density", shrink=0.8)

ax.set_xlabel("X (m)", fontsize=12)
ax.set_ylabel("Y (m)", fontsize=12)
ax.set_title("Spatial Coverage Density", fontsize=14, fontweight="bold")
ax.set_xlim(-14, 5)
ax.set_ylim(-20, 3)
plt.tight_layout()
plt.show()

## 3. IMU Analysis

In [ ]:
# --- IMU: Box plots per axis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

accel_data = df_imu[["accel_x", "accel_y", "accel_z"]].melt(var_name="Axis", value_name="Value")
accel_data["Axis"] = accel_data["Axis"].map({"accel_x": "X", "accel_y": "Y", "accel_z": "Z"})
sns.boxplot(data=accel_data, x="Axis", y="Value", palette="Set2", ax=axes[0],
            showfliers=False, width=0.5)
axes[0].set_title("Accelerometer (m/s²)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("")

gyro_data = df_imu[["gyro_x", "gyro_y", "gyro_z"]].melt(var_name="Axis", value_name="Value")
gyro_data["Axis"] = gyro_data["Axis"].map({"gyro_x": "X", "gyro_y": "Y", "gyro_z": "Z"})
sns.boxplot(data=gyro_data, x="Axis", y="Value", palette="Set3", ax=axes[1],
            showfliers=False, width=0.5)
axes[1].set_title("Gyroscope (rad/s)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("")

fig.suptitle("IMU Distributions — All Paths", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- IMU time series for a single path (interactive) ---
sample_pid = df_stats.loc[df_stats["imu_events"].idxmax(), "path_id"]
imu_sample = df_imu[df_imu["path_id"] == sample_pid].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Accelerometer", "Gyroscope"],
                    vertical_spacing=0.08)

for col, color in [("accel_x", "#1f77b4"), ("accel_y", "#ff7f0e"), ("accel_z", "#2ca02c")]:
    fig.add_trace(go.Scatter(x=imu_sample["sim_time"], y=imu_sample[col],
                             name=col, line=dict(width=1, color=color)), row=1, col=1)

for col, color in [("gyro_x", "#d62728"), ("gyro_y", "#9467bd"), ("gyro_z", "#8c564b")]:
    fig.add_trace(go.Scatter(x=imu_sample["sim_time"], y=imu_sample[col],
                             name=col, line=dict(width=1, color=color)), row=2, col=1)

fig.update_layout(title=f"IMU Time Series — Path {sample_pid} (longest path)",
                  height=500, template="plotly_white",
                  xaxis2_title="Simulation Time (s)")
fig.show()

## 4. Odometry Analysis

In [ ]:
# --- Odometry: velocity profiles ---
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Linear Velocity", "Angular Velocity"],
                    vertical_spacing=0.08)

odom_sample = df_odom[df_odom["path_id"] == sample_pid]

fig.add_trace(go.Scatter(
    x=odom_sample["sim_time"], y=odom_sample["odom_linear_vel"],
    name="Linear vel (m/s)", line=dict(width=1.5, color="#1f77b4")
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=odom_sample["sim_time"], y=odom_sample["odom_angular_vel"],
    name="Angular vel (rad/s)", line=dict(width=1.5, color="#ff7f0e")
), row=2, col=1)

fig.update_layout(title=f"Odometry Velocity Profile — Path {sample_pid}",
                  height=450, template="plotly_white",
                  xaxis2_title="Simulation Time (s)")
fig.show()

In [ ]:
# --- GT vs Odometry (PDR): rotation-corrected, error-colored ---
from scipy.spatial import cKDTree

demo_pid = df_stats.loc[df_stats["gt_events"].idxmax(), "path_id"]
gt_p = df_gt[df_gt["path_id"] == demo_pid].sort_values("sim_time").reset_index(drop=True)
odom_p = df_odom[df_odom["path_id"] == demo_pid].sort_values("sim_time").reset_index(drop=True)

# GT start position and initial heading
gt_start_x, gt_start_y = gt_p["gt_x"].iloc[0], gt_p["gt_y"].iloc[0]
gt_heading_0 = gt_p["gt_heading_rad"].iloc[0]

# Odometry is in the robot's local frame starting at (0,0) heading=0
# The robot's initial heading in world frame is gt_heading_0
# But odom accumulates from heading=0, so we rotate odom by gt_heading_0
# to align it with the world frame
odom_raw_x = odom_p["odom_x"].values
odom_raw_y = odom_p["odom_y"].values

# Rotate odom from robot frame to world frame
cos_h = np.cos(gt_heading_0)
sin_h = np.sin(gt_heading_0)
odom_x_world = cos_h * odom_raw_x - sin_h * odom_raw_y + gt_start_x
odom_y_world = sin_h * odom_raw_x + cos_h * odom_raw_y + gt_start_y

# Compute error at each odom point
gt_tree = cKDTree(gt_p[["gt_x", "gt_y"]].values)
odom_xy = np.column_stack([odom_x_world, odom_y_world])
distances, _ = gt_tree.query(odom_xy)

# --- Plotly side-by-side ---
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"Ground Truth — Path {demo_pid}",
    f"Dead Reckoning (PDR) — Error Colored"
], horizontal_spacing=0.06)

# Walls on both panels
for wt in plotly_wall_traces(showlegend_once=False):
    fig.add_trace(wt.update(fillcolor="rgba(210,210,210,0.4)"), row=1, col=1)
    fig.add_trace(go.Scatter(x=wt.x, y=wt.y, fill="toself",
        fillcolor="rgba(210,210,210,0.4)",
        line=dict(color="rgba(100,100,100,0.6)", width=1),
        showlegend=False, hoverinfo="skip"), row=1, col=2)

# GT trajectory
fig.add_trace(go.Scatter(
    x=gt_p["gt_x"], y=gt_p["gt_y"],
    mode="lines", line=dict(width=3, color="#2196F3"),
    name="Ground Truth",
), row=1, col=1)

# Start/End on GT
fig.add_trace(go.Scatter(
    x=[gt_p["gt_x"].iloc[0]], y=[gt_p["gt_y"].iloc[0]],
    mode="markers", marker=dict(size=12, color="#4CAF50", symbol="circle"),
    name="Start",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[gt_p["gt_x"].iloc[-1]], y=[gt_p["gt_y"].iloc[-1]],
    mode="markers", marker=dict(size=12, color="#F44336", symbol="x"),
    name="End",
), row=1, col=1)

# PDR colored by error — draw in chunks for performance
chunk_size = 20
for j in range(0, len(odom_xy) - chunk_size, chunk_size):
    end = min(j + chunk_size + 1, len(odom_xy))
    avg_err = distances[j:end].mean()
    norm_err = min(avg_err / max(distances.max(), 0.5), 1.0)
    r_c = int(255 * norm_err)
    g_c = int(255 * (1 - norm_err))
    color = f"rgb({r_c},{g_c},50)"

    fig.add_trace(go.Scatter(
        x=odom_xy[j:end, 0], y=odom_xy[j:end, 1],
        mode="lines", line=dict(width=3, color=color),
        showlegend=False,
        hovertemplate=f"Drift: {avg_err:.2f}m<extra></extra>",
    ), row=1, col=2)

# Start/End on PDR
fig.add_trace(go.Scatter(
    x=[odom_xy[0, 0]], y=[odom_xy[0, 1]],
    mode="markers", marker=dict(size=12, color="#4CAF50", symbol="circle"),
    showlegend=False,
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=[odom_xy[-1, 0]], y=[odom_xy[-1, 1]],
    mode="markers", marker=dict(size=12, color="#F44336", symbol="x"),
    showlegend=False,
), row=1, col=2)

# Auto-scale to include both GT and PDR
all_x = np.concatenate([gt_p["gt_x"].values, odom_xy[:, 0]])
all_y = np.concatenate([gt_p["gt_y"].values, odom_xy[:, 1]])
x_pad = (all_x.max() - all_x.min()) * 0.1 + 1
y_pad = (all_y.max() - all_y.min()) * 0.1 + 1
x_range = [all_x.min() - x_pad, all_x.max() + x_pad]
y_range = [all_y.min() - y_pad, all_y.max() + y_pad]

fig.update_xaxes(range=x_range, showgrid=False, zeroline=False,
                 showline=True, linewidth=1.5, linecolor="#888", mirror=True)
fig.update_yaxes(range=y_range, showgrid=False, zeroline=False, scaleanchor="x",
                 showline=True, linewidth=1.5, linecolor="#888", mirror=True)
fig.update_layout(
    title=f"Ground Truth vs Dead Reckoning — Path {demo_pid}<br>"
          f"<sub>Max drift: {distances.max():.2f}m | Mean drift: {distances.mean():.2f}m | "
          f"Initial heading: {_math.degrees(gt_heading_0):.1f}°</sub>",
    height=600, width=1100,
    template="plotly_white",
    plot_bgcolor="white",
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 5. WiFi RSSI Analysis

In [ ]:
# --- WiFi: RSSI spatial map for top 4 APs (Plotly) ---
rssi_cols = [c for c in df_wifi.columns if c.startswith("wifi_rssi_")]
mean_rssi = df_wifi[rssi_cols].replace(-200.0, np.nan).mean().sort_values(ascending=False)
top4 = mean_rssi.head(4).index.tolist()

# Match each WiFi scan to nearest GT position
wifi_rows = []
for pid in df_wifi["path_id"].unique():
    w = df_wifi[df_wifi["path_id"] == pid]
    g = df_gt[df_gt["path_id"] == pid]
    if g.empty:
        continue
    gt_times = g["sim_time"].values
    gt_x = g["gt_x"].values
    gt_y = g["gt_y"].values
    for _, row in w.iterrows():
        idx = np.argmin(np.abs(gt_times - row["sim_time"]))
        row_dict = row.to_dict()
        row_dict["gt_x"] = gt_x[idx]
        row_dict["gt_y"] = gt_y[idx]
        wifi_rows.append(row_dict)
wifi_with_pos = pd.DataFrame(wifi_rows)

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f"AP ...{c.replace('wifi_rssi_','')[-8:]}" for c in top4],
                    horizontal_spacing=0.06, vertical_spacing=0.08)

for i, ap_col in enumerate(top4):
    r = i // 2 + 1
    c = i % 2 + 1

    for wt in plotly_wall_traces(showlegend_once=False):
        fig.add_trace(wt.update(fillcolor="rgba(220,220,220,0.35)"), row=r, col=c)

    rssi_vals = wifi_with_pos[ap_col].replace(-200.0, np.nan)
    mask = rssi_vals.notna()

    fig.add_trace(go.Scatter(
        x=wifi_with_pos.loc[mask, "gt_x"],
        y=wifi_with_pos.loc[mask, "gt_y"],
        mode="markers",
        marker=dict(
            size=7, color=rssi_vals[mask], colorscale="RdYlGn",
            cmin=-95, cmax=-30, opacity=0.85,
            colorbar=dict(title="dBm", len=0.4, y=0.8 - 0.5 * (i // 2), x=1.02) if c == 2 else None,
        ),
        showlegend=False,
        hovertemplate="x=%{x:.1f}<br>y=%{y:.1f}<br>RSSI=%{marker.color:.0f} dBm<extra></extra>",
    ), row=r, col=c)

fig.update_xaxes(range=[-14, 5], showgrid=False, zeroline=False, showticklabels=False,
                 showline=True, linewidth=1.5, linecolor="#888", mirror=True)
fig.update_yaxes(range=[-20, 3], showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x",
                 showline=True, linewidth=1.5, linecolor="#888", mirror=True)
fig.update_layout(
    title_text="WiFi RSSI Spatial Maps — 4 Strongest Access Points",
    title_font_size=16,
    height=900, width=900,
    template="plotly_white",
    plot_bgcolor="white",
)
fig.show()

In [ ]:
# --- WiFi: visible APs and strongest RSSI over time ---
fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=["Visible APs per Scan (RSSI > -85 dBm)",
                                    "Strongest RSSI per Scan"])

for pid in df_stats["path_id"][:4]:
    w = df_wifi[df_wifi["path_id"] == pid]
    fig.add_trace(go.Scatter(
        x=w["sim_time"], y=w["wifi_visible_count"],
        name=f"Path {pid}", mode="lines+markers", marker=dict(size=3)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=w["sim_time"], y=w["wifi_strongest_rssi"],
        name=f"Path {pid}", mode="lines+markers", marker=dict(size=3),
        showlegend=False
    ), row=2, col=1)

fig.update_yaxes(title_text="Visible APs", row=1, col=1)
fig.update_yaxes(title_text="RSSI (dBm)", row=2, col=1)
fig.update_xaxes(title_text="Simulation Time (s)", row=2, col=1)
fig.update_layout(title="WiFi Signal Characteristics", height=500, template="plotly_white")
fig.show()

## 6. Async Timing Analysis
One of the key features of this dataset: **sensors fire independently at jittered rates**, simulating real-world asynchronous multi-modal acquisition.

In [ ]:
# --- Async rates: actual vs nominal (simple bar comparison) ---
nominal_rates = {"IMU": 31.25, "Odometry": 15.625, "Ground Truth": 10.42, "WiFi": 1.0, "Camera": 0.5}

actual_rates = {}
for name, df, tcol in [("IMU", df_imu, "sim_time"), ("Odometry", df_odom, "sim_time"),
                         ("Ground Truth", df_gt, "sim_time"), ("WiFi", df_wifi, "sim_time"),
                         ("Camera", df_cam, "sim_time")]:
    dts = []
    for pid in df["path_id"].unique():
        t = df[df["path_id"] == pid][tcol].sort_values().diff().dropna()
        dts.append(t)
    all_dt = pd.concat(dts)
    actual_rates[name] = 1.0 / all_dt.median() if all_dt.median() > 0 else 0

rate_df = pd.DataFrame({
    "Modality": list(nominal_rates.keys()),
    "Nominal (Hz)": list(nominal_rates.values()),
    "Actual Median (Hz)": [actual_rates[m] for m in nominal_rates],
})

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(rate_df))
w = 0.35
bars1 = ax.bar(x - w/2, rate_df["Nominal (Hz)"], w, label="Nominal", color="#90CAF9", edgecolor="#1565C0")
bars2 = ax.bar(x + w/2, rate_df["Actual Median (Hz)"], w, label="Actual (median)", color="#A5D6A7", edgecolor="#2E7D32")

ax.set_xticks(x)
ax.set_xticklabels(rate_df["Modality"], fontsize=11)
ax.set_ylabel("Frequency (Hz)", fontsize=12)
ax.set_title("Nominal vs Actual Sensor Rates (±20% jitter)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)

# Add rate labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}", ha="center", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}", ha="center", fontsize=9)

ax.set_yscale("log")
ax.set_ylim(0.3, 60)
plt.tight_layout()
plt.show()

In [ ]:
# --- Multi-modal timeline: zoomed to a 10-second window ---
timeline_pid = sample_pid
t_all = df_imu[df_imu["path_id"] == timeline_pid]["sim_time"]
t_mid = t_all.median()
t_start, t_end = t_mid - 5, t_mid + 5  # 10s window

fig = go.Figure()

modality_data = [
    ("IMU", df_imu, 5, "#1f77b4"),
    ("Odometry", df_odom, 4, "#ff7f0e"),
    ("Ground Truth", df_gt, 3, "#2ca02c"),
    ("WiFi", df_wifi, 2, "#d62728"),
    ("Camera", df_cam, 1, "#9467bd"),
]

for name, df, y_pos, color in modality_data:
    t = df[(df["path_id"] == timeline_pid) & (df["sim_time"] >= t_start) & (df["sim_time"] <= t_end)]["sim_time"]
    fig.add_trace(go.Scatter(
        x=t, y=[y_pos] * len(t),
        mode="markers", name=f"{name} ({len(t)} events)",
        marker=dict(size=8, color=color, symbol="line-ns-open", line=dict(width=2)),
    ))

fig.update_layout(
    title=f"Async Event Timeline — Path {timeline_pid} (10s window: {t_start:.0f}–{t_end:.0f}s)",
    yaxis=dict(tickvals=[1, 2, 3, 4, 5],
               ticktext=["Camera", "WiFi", "GT", "Odom", "IMU"]),
    xaxis_title="Simulation Time (s)",
    height=350, template="plotly_white",
    showlegend=True
)
fig.show()

## 7. Camera Samples

In [ ]:
# --- Camera: sample RGB + Depth pairs from different paths ---
from PIL import Image

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for row, pid in enumerate(df_stats["path_id"][:3]):
    cam_dir = DATA_DIR / f"path_{pid:02d}" / "camera"
    rgb_files = sorted(cam_dir.glob("rgb_*.png"))
    depth_files = sorted(cam_dir.glob("depth_*.png"))
    
    if len(rgb_files) < 2:
        continue
    
    # Pick 2 evenly spaced frames
    indices = [len(rgb_files) // 4, 3 * len(rgb_files) // 4]
    
    for col_offset, idx in enumerate(indices):
        rgb_img = Image.open(rgb_files[idx])
        depth_img = Image.open(depth_files[idx])
        
        axes[row, col_offset * 2].imshow(rgb_img)
        axes[row, col_offset * 2].set_title(f"Path {pid} — RGB #{idx}", fontsize=10)
        axes[row, col_offset * 2].axis("off")
        
        axes[row, col_offset * 2 + 1].imshow(depth_img, cmap="inferno")
        axes[row, col_offset * 2 + 1].set_title(f"Path {pid} — Depth #{idx}", fontsize=10)
        axes[row, col_offset * 2 + 1].axis("off")

fig.suptitle("Camera Samples — RGB and Depth Pairs", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Per-Path Statistics

In [ ]:
# --- Per-path stats table ---
display_stats = df_stats.copy()
display_stats.columns = ["Path", "Duration (s)", "IMU", "Odometry", "WiFi", "Ground Truth",
                          "Camera", "RGB Images", "Disk (MB)"]
display_stats.style.background_gradient(cmap="Blues", subset=["IMU", "Odometry", "WiFi", "Ground Truth", "Camera"])\
    .format({"Duration (s)": "{:.0f}", "Disk (MB)": "{:.0f}"})

## 9. Final Summary & Projections

> **Note:** Collection is still in progress. Values below are **projected** from the collected paths to the full 30-path dataset.

In [ ]:
# --- Final projection summary ---
total_events = len(df_imu) + len(df_odom) + len(df_wifi) + len(df_gt) + len(df_cam)
total_images = df_stats["rgb_images"].sum() * 2  # RGB + Depth
total_duration = df_stats["duration_s"].sum()

print("=" * 60)
print("  NavLoRI Async Data Collection — Summary")
print("=" * 60)
print(f"\n  Status: {N_COLLECTED}/{TOTAL_PLANNED_PATHS} paths collected (simulation ongoing)")
print(f"\n  ── Collected ({N_COLLECTED} paths) ──")
print(f"  Total sensor events:    {total_events:>10,}")
print(f"  Total images (RGB+D):   {total_images:>10,}")
print(f"  Total sim duration:     {total_duration:>10,.0f} s ({total_duration/60:.0f} min)")
print(f"  Disk usage:             {df_stats['disk_mb'].sum():>10,.0f} MB")
print(f"\n  ── Projected (30 paths) ──")
print(f"  Total sensor events:    {int(total_events * scale):>10,}")
print(f"  Total images (RGB+D):   {int(total_images * scale):>10,}")
print(f"  Total sim duration:     {total_duration * scale:>10,.0f} s ({total_duration * scale/60:.0f} min)")
print(f"  Disk usage:             ~{df_stats['disk_mb'].sum() * scale / 1000:>9.1f} GB")
print(f"\n  ── Key Properties ──")
print(f"  Modalities:             5 (IMU, Odometry, WiFi, Camera, GT)")
print(f"  WiFi access points:     117")
print(f"  Async jitter:           ±20% on all sensor intervals")
print(f"  Camera resolution:      640×480 (RGB + Depth)")
print(f"  Environment:            Indoor (CESI LINEACT sim)")
print("=" * 60)